# 🌳 Ensemble Methods Comparison

> **Master ensemble learning with Random Forest, XGBoost, and advanced techniques**

This notebook provides comprehensive coverage of ensemble methods, comparing their strengths, weaknesses, and optimal use cases.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Understand** different ensemble strategies (bagging, boosting, stacking)
- **Master** Random Forest and XGBoost implementations
- **Compare** ensemble methods systematically
- **Optimize** hyperparameters effectively
- **Build** custom ensemble models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✅ All imports successful!")

## 🌳 Ensemble Methods Comparison Framework

In [ ]:
class EnsembleComparison:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.models = {}
        self.results = {}
    
    def add_model(self, name, model):
        """Add a model to comparison"""
        self.models[name] = model
    
    def train_all_models(self):
        """Train all models and collect results"""
        print("🚀 Training all ensemble models...\n")
        
        for name, model in self.models.items():
            print(f"Training {name}...")
            
            # Train model
            model.fit(self.X_train, self.y_train)
            
            # Make predictions
            train_pred = model.predict(self.X_train)
            test_pred = model.predict(self.X_test)
            
            # Calculate metrics
            train_acc = accuracy_score(self.y_train, train_pred)
            test_acc = accuracy_score(self.y_test, test_pred)
            
            # Store results
            self.results[name] = {
                'model': model,
                'train_accuracy': train_acc,
                'test_accuracy': test_acc,
                'overfitting': train_acc - test_acc,
                'train_predictions': train_pred,
                'test_predictions': test_pred
            }
            
            print(f"  Train Accuracy: {train_acc:.4f}")
            print(f"  Test Accuracy: {test_acc:.4f}")
            print(f"  Overfitting: {train_acc - test_acc:.4f}\n")
    
    def plot_comparison(self):
        """Plot model comparison"""
        if not self.results:
            print("No results available. Train models first.")
            return
        
        # Prepare data for plotting
        models = list(self.results.keys())
        train_accs = [self.results[m]['train_accuracy'] for m in models]
        test_accs = [self.results[m]['test_accuracy'] for m in models]
        overfitting = [self.results[m]['overfitting'] for m in models]
        
        # Create subplots
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Accuracy comparison
        x = np.arange(len(models))
        width = 0.35
        
        axes[0].bar(x - width/2, train_accs, width, label='Train', alpha=0.8)
        axes[0].bar(x + width/2, test_accs, width, label='Test', alpha=0.8)
        axes[0].set_xlabel('Models')
        axes[0].set_ylabel('Accuracy')
        axes[0].set_title('Train vs Test Accuracy')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(models, rotation=45)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Overfitting analysis
        colors = ['red' if x > 0.05 else 'green' for x in overfitting]
        axes[1].bar(models, overfitting, color=colors, alpha=0.7)
        axes[1].axhline(y=0.05, color='red', linestyle='--', label='Overfitting threshold')
        axes[1].set_xlabel('Models')
        axes[1].set_ylabel('Train - Test Accuracy')
        axes[1].set_title('Overfitting Analysis')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # Test accuracy ranking
        sorted_models = sorted(zip(models, test_accs), key=lambda x: x[1], reverse=True)
        sorted_names, sorted_accs = zip(*sorted_models)
        
        axes[2].barh(range(len(sorted_names)), sorted_accs, alpha=0.8)
        axes[2].set_yticks(range(len(sorted_names)))
        axes[2].set_yticklabels(sorted_names)
        axes[2].set_xlabel('Test Accuracy')
        axes[2].set_title('Model Ranking (Test Accuracy)')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def get_best_model(self):
        """Get the best performing model"""
        if not self.results:
            return None
        
        best_model_name = max(self.results.keys(), 
                            key=lambda x: self.results[x]['test_accuracy'])
        
        return best_model_name, self.results[best_model_name]

print("✅ EnsembleComparison class defined!")

In [ ]:
# Generate sample data
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, 
                          n_redundant=5, n_clusters_per_class=2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Initialize comparison framework
comparison = EnsembleComparison(X_train, X_test, y_train, y_test)

# Add models to compare
comparison.add_model('Decision Tree', DecisionTreeClassifier(random_state=42))
comparison.add_model('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42))
comparison.add_model('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42))
comparison.add_model('XGBoost', xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'))

# Train all models
comparison.train_all_models()

# Plot comparison
comparison.plot_comparison()

# Get best model
best_name, best_result = comparison.get_best_model()
print(f"🏆 Best model: {best_name} (Test Accuracy: {best_result['test_accuracy']:.4f})")

## ⚙️ Hyperparameter Optimization

In [ ]:
def optimize_random_forest(X_train, y_train):
    """Optimize Random Forest hyperparameters"""
    
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    rf = RandomForestClassifier(random_state=42)
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    
    print("🔍 Optimizing Random Forest...")
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

def optimize_xgboost(X_train, y_train):
    """Optimize XGBoost hyperparameters"""
    
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 0.9, 1.0]
    }
    
    xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
    grid_search = GridSearchCV(xgb_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    
    print("🔍 Optimizing XGBoost...")
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

# Optimize models
best_rf = optimize_random_forest(X_train, y_train)
best_xgb = optimize_xgboost(X_train, y_train)

# Compare optimized models
optimized_comparison = EnsembleComparison(X_train, X_test, y_train, y_test)
optimized_comparison.add_model('Optimized Random Forest', best_rf)
optimized_comparison.add_model('Optimized XGBoost', best_xgb)

optimized_comparison.train_all_models()
optimized_comparison.plot_comparison()

## 🎯 Practice Problems

### **Problem 1: Custom Ensemble Model**
Create a custom ensemble that combines multiple base models.

In [ ]:
class CustomEnsemble:
    def __init__(self, base_models, voting='soft'):
        """
        Custom ensemble classifier
        
        Parameters:
        base_models: list of (name, model) tuples
        voting: 'hard' or 'soft' voting
        """
        self.base_models = base_models
        self.voting = voting
        self.trained_models = []
    
    def fit(self, X, y):
        """Train all base models"""
        # Your code here
        pass
    
    def predict(self, X):
        """Make ensemble predictions"""
        # Your code here
        pass
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        # Your code here
        pass

# Test your ensemble
# base_models = [
#     ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
#     ('xgb', xgb.XGBClassifier(n_estimators=50, random_state=42)),
#     ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42))
# ]
# 
# custom_ensemble = CustomEnsemble(base_models, voting='soft')
# custom_ensemble.fit(X_train, y_train)

## 🎯 Key Takeaways

1. **Ensemble methods** often outperform individual models
2. **Random Forest** is robust and handles overfitting well
3. **XGBoost** often achieves the best performance with tuning
4. **Hyperparameter optimization** is crucial for ensemble methods
5. **Custom ensembles** can combine strengths of different algorithms

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Try stacking ensembles** with meta-learners
3. **Move to the next notebook**: Deep Learning

---

**Excellent ensemble skills!** 🎉 You can now build powerful ensemble models for any ML problem.